In [1]:
import pandas as pd

df = pd.read_parquet("data/train/legal_corpus.parquet", engine="pyarrow")
df

,id,law_id,aid,content_Article
0,0,14/2022/TT-NHNN,0,"1. Thông tư này quy định mã số, tiêu chuẩn chu..."
1,0,14/2022/TT-NHNN,1,1. Kiểm soát viên cao cấp ngân hàng Mã số: 07....
2,0,14/2022/TT-NHNN,2,"1. Có bản lĩnh chính trị vững vàng, kiên định ..."
3,0,14/2022/TT-NHNN,3,1. Chức trách Là công chức có trình độ chuyên ...
4,0,14/2022/TT-NHNN,4,1. Chức trách Là công chức có trình độ chuyên ...
...,...,...,...,...
59631,2154,72/2019/NĐ-CP,59631,"Các Bộ trưởng, Thủ trưởng cơ quan ngang bộ, Th..."
59632,2155,18/2023/NĐ-CP,59632,"1. Sửa\r\nđổi, bổ sung khoản 2 Điều 3\nnhư sau..."
59633,2155,18/2023/NĐ-CP,59633,1. Bộ trưởng Bộ Công Thương chịu trách nhiệm t...
59634,2155,18/2023/NĐ-CP,59634,1. Nghị định này có hiệu lực thi hành từ ngày ...


In [2]:
law_ids = list(df["law_id"].unique())
print(len(law_ids))

2156


In [3]:
from src.preprocess.get_law_url import search_law_ids
import os
from dotenv import load_dotenv
load_dotenv()

# search_law_ids(api_key=os.getenv("SEARCH_API"), law_ids=law_ids)

True

In [4]:
from src.utils import normalize_general
import json
with open("data/search_result.json", "r", encoding="utf-8") as f:
    result = json.load(f)

law_id_not_found = []
data = {}
for law_id, value in result.items():
    search_result = value["organic"]
    is_found = False
    for item in search_result:
        item["title"] = normalize_general(item["title"])
        if item["link"].startswith("https://thuvienphapluat.vn/") and item["link"].endswith(".aspx") and law_id.lower().replace(" ", "") in item["title"].lower():
            data[law_id] = item["link"]
            is_found = True
            break
    if not is_found:
        law_id_not_found.append(law_id)
        print(f"❌ Không tìm thấy URL cho {law_id}")

print(len(data))

❌ Không tìm thấy URL cho 49/2005/QH11
❌ Không tìm thấy URL cho 52/2010/QH12
❌ Không tìm thấy URL cho 13/2011/TT-BTP
❌ Không tìm thấy URL cho 43/2011/TT-BCA
❌ Không tìm thấy URL cho 18/2011/TT-BKHCN
❌ Không tìm thấy URL cho 182/2009/TT-BTC
❌ Không tìm thấy URL cho 18/2022/TT-BYT
❌ Không tìm thấy URL cho 03/2007/QH12
❌ Không tìm thấy URL cho 03/2010/TTLT-BYT-BCA
❌ Không tìm thấy URL cho KhongSo
❌ Không tìm thấy URL cho 1332/QĐ-BGTVT
❌ Không tìm thấy URL cho 33/2018/TT-BGTVT
❌ Không tìm thấy URL cho 68/2014/QH13
❌ Không tìm thấy URL cho 128-QĐ/TW
❌ Không tìm thấy URL cho 06/2012/TT-BNV
❌ Không tìm thấy URL cho 08/TT
❌ Không tìm thấy URL cho 24-NQ/TW
❌ Không tìm thấy URL cho 04/2023/QĐ-TTg
❌ Không tìm thấy URL cho 08/2012/TT-BXD
❌ Không tìm thấy URL cho 27/2008/QH12
❌ Không tìm thấy URL cho 39/2011/TT-BCT
❌ Không tìm thấy URL cho 43/2011/NĐ-CP
❌ Không tìm thấy URL cho 17/2016/TT-BTTTT
❌ Không tìm thấy URL cho 157/2007/QĐ-TTg
❌ Không tìm thấy URL cho 63/2022/TT-BQP
❌ Không tìm thấy URL cho 

In [5]:
import pandas as pd
df = pd.DataFrame({"law_id": law_id_not_found})
df["url"] = ""
df.to_excel("data/law_id_not_found.xlsx", index=False)
df

,law_id,url
0,49/2005/QH11,
1,52/2010/QH12,
2,13/2011/TT-BTP,
3,43/2011/TT-BCA,
4,18/2011/TT-BKHCN,
5,182/2009/TT-BTC,
6,18/2022/TT-BYT,
7,03/2007/QH12,
8,03/2010/TTLT-BYT-BCA,
9,KhongSo,


In [6]:
import pandas as pd
df = pd.read_excel("data/law_id_additional.xlsx")
for index, row in df.iterrows():
    law_id = row["law_id"]
    url = row["url"]
    if pd.isna(url):
        print(f"❌ {law_id} không có URL")
        continue
    if law_id in data:
        print(f"❌ {law_id} đã có trong data")
    else:
        data[law_id] = url
        print(f"✅ {law_id}")

print(len(data))

✅ 49/2005/QH11
✅ 52/2010/QH12
✅ 13/2011/TT-BTP
✅ 43/2011/TT-BCA
✅ 18/2011/TT-BKHCN
✅ 182/2009/TT-BTC
✅ 18/2022/TT-BYT
✅ 03/2007/QH12
✅ 03/2010/TTLT-BYT-BCA
✅ KhongSo
✅ 1332/QĐ-BGTVT
✅ 33/2018/TT-BGTVT
✅ 68/2014/QH13
✅ 128-QĐ/TW
✅ 06/2012/TT-BNV
✅ 08/TT
✅ 24-NQ/TW
✅ 04/2023/QĐ-TTg
✅ 08/2012/TT-BXD
✅ 27/2008/QH12
✅ 39/2011/TT-BCT
✅ 43/2011/NĐ-CP
✅ 17/2016/TT-BTTTT
✅ 157/2007/QĐ-TTg
✅ 63/2022/TT-BQP
✅ 24/QĐ-VKSTC
✅ 66/2020/QH14
✅ 73/2009/NĐ-CP
✅ 01/2010/TT-BNG
✅ 90/2012/NĐ-CP
✅ 08/2023/QĐ-TTg
✅ 56/2011/NĐ-CP
✅ 26/2013/TT-BYT
✅ 16/2013/TTLT-BYT-BNN&PTNT
✅ 66/2023/TT-BCA
✅ 35/2018/QH14
✅ 52/2019/QH14
✅ 28/2018/QH14
✅ 18/2018/TT-BTP
✅ 85/2015/QH13
✅ 05/2024/TT-BLDTBXH
2156


In [7]:
with open("data/law_urls.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)